In [33]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Ridge, Lasso

In [34]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import r2_score, mean_absolute_percentage_error
from sklearn.model_selection import cross_val_score, KFold

# <center> Подбор гиперпараметров </center>

In [35]:
df = pd.read_csv("https://raw.githubusercontent.com/katarina74/ml_lessons/main/lesson_2/data/techparams_train.csv")
X = df.drop(["target", "index"], axis=1)
y = df[["target"]]

ordinal = ['back-suspension', 'cylinders-order', 'engine-feeding', 'configurations_seats']
numerical = ['battery-capacity', 'charge-time', 'compression', 'consumption-mixed', 'cylinders-value', 'engine-start', 'engine-stop', 'luxury',
             'max-speed', 'power-electro-kw','valves','weight','configurations_auto-premiere','configurations_back-wheel-base','configurations_tank-volume',
             'supergen_year-stop']
catigorial = ['engine-start','supergen_year-stop','engine-stop','engine-type','gear-type', 'supply-system', 'valvetrain', 'configurations_front-brake', 'configurations_safety-rating','models_country-from',
              'models_group', 'models_light-and-commercial','models_male']

X_crop = X.drop(catigorial, axis=1)

train_X, test_X, train_y, test_y = train_test_split(X_crop, y, test_size=0.33, random_state=42)

In [36]:
train_X.info()

<class 'pandas.core.frame.DataFrame'>
Index: 28974 entries, 22607 to 15795
Data columns (total 17 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   back-suspension                 28974 non-null  int64  
 1   battery-capacity                28974 non-null  float64
 2   charge-time                     28974 non-null  int64  
 3   compression                     28974 non-null  float64
 4   consumption-mixed               28974 non-null  float64
 5   cylinders-order                 28974 non-null  int64  
 6   cylinders-value                 28974 non-null  int64  
 7   engine-feeding                  28974 non-null  int64  
 8   luxury                          28974 non-null  int64  
 9   max-speed                       28974 non-null  float64
 10  power-electro-kw                28974 non-null  float64
 11  valves                          28974 non-null  int64  
 12  weight                          2

In [37]:
train_X.head()

,back-suspension,battery-capacity,charge-time,compression,consumption-mixed,cylinders-order,cylinders-value,engine-feeding,luxury,max-speed,power-electro-kw,valves,weight,configurations_auto-premiere,configurations_back-wheel-base,configurations_seats,configurations_tank-volume
22607,3,-1.0,11636,9.5,-1.0,0,4,4,11923,160.0,-1.0,2,1290.0,11642,1415.0,13,50.0
36531,8,-1.0,31764,9.0,-1.0,0,4,0,31915,145.0,-1.0,0,935.0,31785,1310.0,13,48.0
24048,8,-1.0,2773,8.8,6.8,0,4,4,2258,221.0,-1.0,2,1373.0,2763,1558.0,13,56.0
34819,3,-1.0,52039,14.0,5.3,0,4,5,51108,198.0,-1.0,2,1430.0,52081,1590.0,13,56.0
1328,3,-1.0,20689,18.0,7.5,0,5,5,20228,195.0,-1.0,2,1485.0,20685,1560.0,13,70.0


In [38]:
reg = LinearRegression().fit(train_X, train_y)

In [39]:
pred_test = reg.predict(test_X)

In [40]:
mal = mean_absolute_percentage_error(test_y, pred_test)
mal

0.04764777849052345

### Кросс валидация

In [41]:
model = LinearRegression()

In [42]:
scores = cross_val_score(model, train_X, train_y, cv = 10, scoring = 'neg_mean_absolute_percentage_error', verbose = 1)

In [43]:
print(scores)
(scores.mean(), scores.var())

[-0.04719492 -0.04804466 -0.0469994  -0.04572128 -0.04629913 -0.04656426
 -0.04794563 -0.04742356 -0.04580326 -0.04643183]


(np.float64(-0.046842792644124956), np.float64(6.027032806666014e-07))

In [44]:
kf = KFold(n_splits = 10, shuffle = True, random_state = 42)

In [45]:
fold_n = 1
for train_index, test_index in kf.split(train_X, train_y):
    model = LinearRegression()
    model.fit(train_X.iloc[train_index], train_y.iloc[train_index])
    print(fold_n, mean_absolute_percentage_error(train_y.iloc[test_index], model.predict(train_X.iloc[test_index])[:,0]))
    

1 0.04746964159240838
1 0.0481412382843881
1 0.04758550053743821
1 0.04619072877658265
1 0.047383952784324
1 0.04663371718321347
1 0.04602221993114318
1 0.04642143799809981
1 0.047010522553462275
1 0.04551843224293318


### Подбор параметров модели

**Задание 1**

Реализовать подбор по сетке: 
1. Выбрать модель, которую будем улучшать (Ridge, Lasso или ElasticNet).
2. Разбить исходные данные на подгруппы для обучения.
3. Выбрать 1 гиперпараметр, значения которого необходимо подобрать. Определить диапазон значений, которые будем перебирать.
4. Для каждого значения из выбранного диапазона:
    - Обучить модель на каждой подгруппе.
    - Получить значение метрики на тесте.
    - Получить время обучения.
    - Усреднить метрику и время обучения по группам.
5. Выбрать то значение из дипазона, для которого усредненная метрика качества модели будет наилучшей.

In [46]:
import time 
from sklearn.model_selection import KFold

In [47]:
kf = KFold(n_splits=5, shuffle=True, random_state=42) # разбиваем на 5 подгрупп

В качестве гиперпараметра, который мы будем менять, выберем lambda - коэффициент регуляризации, который позволяет более равномерно распределять веса между признаками и не давать больший вес признаку из-за его больших значений, то есть накладывает дополнительные штрафы

In [48]:
lambdas = [0.001, 0.01, 0.1, 1, 10, 100, 1000] # значения для перебора
results = {}

for lam in lambdas:
    mape_scores = []
    times = []
    
    for train_idx, val_idx in kf.split(train_X):
       
        X_train_fold = train_X.iloc[train_idx]  
        X_val_fold = train_X.iloc[val_idx]
        y_train_fold = train_y.iloc[train_idx]
        y_val_fold = train_y.iloc[val_idx]
        
        start_time = time.time()
        
        model = Ridge(alpha=lam)   
        model.fit(X_train_fold, y_train_fold)
        
        end_time = time.time()
        training_time = end_time - start_time
        
        y_pred = model.predict(X_val_fold)

        mape = mean_absolute_percentage_error(y_val_fold, y_pred)
        
        mape_scores.append(mape)
        times.append(training_time)
    
    avg_mape = sum(mape_scores) / len(mape_scores)
    avg_time = sum(times) / len(times)
    
    results[lam] = {'mape': avg_mape, 'time': avg_time}

best_lambda = min(results, key=lambda x: results[x]['mape'])
best_lambda

1000

Получилось, что lambda = 1000 наилучшее

**Задание 2**

Используя функцию GridSearchCV подобрать гиперпараметры для Ridge, Lasso или ElasticNet. Сравнить качество улучшенной модели с LinearRegression.

In [49]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'alpha': [0.001, 0.01, 0.1, 1, 10, 100, 1000]
}

grid_search = GridSearchCV(
    estimator=Ridge(),  # явно указываем модель Ridge
    param_grid=param_grid,
    cv=5,
    scoring='neg_mean_absolute_percentage_error',
    n_jobs=-1,
)

grid_search.fit(train_X, train_y)

print('Лучшие параметры:', grid_search.best_params_)
print('Лучший MAPE:', -grid_search.best_score_)

best_model = grid_search.best_estimator_

pred_test_ridge = best_model.predict(test_X)

mape_ridge = mean_absolute_percentage_error(test_y, pred_test_ridge)
print(f'\nMAPE Ridge с GridSearchCV: {mape_ridge:.6f}')

print(f'\nMAPE LinearRegression: {mal:.6f}') # взяла значение выше
print(f'\nУлучшение: {((mal - mape_ridge) / mal * 100):.2f}%')

Лучшие параметры: {'alpha': 1000}
Лучший MAPE: 0.04683631833862756

MAPE Ridge с GridSearchCV: 0.047658

MAPE LinearRegression: 0.047648

Улучшение: -0.02%
